In [ ]:
# In[1]:

import pandas as pd

# Load the metric_container.csv file for March 10, 2021
file_path = 'dataset/Bank/telemetry/2021_03_10/metric/metric_container.csv'
metric_container_df = pd.read_csv(file_path)

# Extract unique KPI names and their corresponding components (cmdb_id)
kpi_component_mapping = metric_container_df[['kpi_name', 'cmdb_id']].drop_duplicates()

kpi_component_mapping

```
Out[1]:
```
The `metric_container.csv` file for March 10, 2021, contains 1,701 unique KPI names associated with various components. These KPIs include metrics related to CPU utilization, request counts, processing times, error counts, and container-specific metrics. Components such as Tomcat03, Tomcat04, and dockerB2 are among those listed. For a complete understanding, further inspection of the full dataset is recommended.

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name   cmdb_id
0                              OSLinux-CPU_CPU_CPUCpuUtil  Tomcat04
1       Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  Tomcat03
2       Tomcat-Requests_7441-"http-nio-8003"_Processin...  Tomcat03
3       Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  Tomcat03
4       Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  Tomcat03
...                                                   ...       ...
711644  Container-DOCKER_CONTAINER_d27123361435--bcou-...  dockerB2
711645  Container-DOCKER_CONTAINER_d27123361435--bcou-...  dockerB2
711646  Container-DOCKER_CONTAINER_d27123361435--bcou-...  dockerB2
711647  Container-DOCKER_CONTAINER_d27123361435--bcou-...  dockerB2
711648  Container-DOCKER_CONTAINER_d27123361435--bcou-...  dockerB2

[1701 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Filter the data for possible root cause components
possible_components = [
    "apache01", "apache02", "Tomcat01", "Tomcat02", "Tomcat03", "Tomcat04",
    "MG01", "MG02", "IG01", "IG02", "Mysql01", "Mysql02", "Redis01", "Redis02"
]
filtered_df = metric_container_df[metric_container_df['cmdb_id'].isin(possible_components)]

# Create a new column for 'component-KPI' combination
filtered_df['component_KPI'] = filtered_df['cmdb_id'] + '-' + filtered_df['kpi_name']

# Calculate the global P95 threshold for each 'component-KPI' time series
p95_thresholds = filtered_df.groupby('component_KPI')['value'].quantile(0.95).reset_index()
p95_thresholds.columns = ['component_KPI', 'P95_Threshold']

p95_thresholds

```
Out[2]:
```
The `metric_container.csv` file for March 10, 2021, contains 1,651 unique 'component-KPI' combinations for the possible root cause components. For each 'component-KPI' time series, the global P95 threshold has been calculated. These thresholds represent the 95th percentile of the values for each metric, providing a benchmark for identifying potential anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

component_KPI  P95_Threshold
0         IG01-JVM-Memory_7778_JVM_Memory_HeapMemoryMax   2.040136e+09
1       IG01-JVM-Memory_7778_JVM_Memory_HeapMemoryUsage   4.186372e+01
2        IG01-JVM-Memory_7778_JVM_Memory_HeapMemoryUsed   8.511027e+08
3      IG01-JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed   5.793089e+07
4        IG01-JVM-Operating System_7778_JVM_JVM_CPULoad   5.267200e-01
...                                                 ...            ...
1646              apache02-OSLinux-OSLinux_SWAP_SWAP_Si   0.000000e+00
1647              apache02-OSLinux-OSLinux_SWAP_SWAP_So   0.000000e+00
1648  apache02-OSLinux-OSLinux_SYSTEM_SYSTEM_Check-D...   1.000000e+00
1649  apache02-OSLinux-OSLinux_SYSTEM_SYSTEM_Check-H...   1.000000e+00
1650        apache02-OSLinux-OSLinux_ZABBIX_Host_Uptime   6.586319e+06

[1651 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pytz
from datetime import datetime

# Define the time range in UTC+8
timezone = pytz.timezone('Asia/Shanghai')
start_time = timezone.localize(datetime(2021, 3, 10, 15, 0)).timestamp()
end_time = timezone.localize(datetime(2021, 3, 10, 15, 30)).timestamp()

# Filter the data within the specified time range
filtered_time_range_df = filtered_df[
    (filtered_df['timestamp'] >= start_time) & (filtered_df['timestamp'] <= end_time)
]

filtered_time_range_df

```
Out[3]:
```
The filtered data for March 10, 2021, from 15:00 to 15:30 includes 26,456 rows, representing the relevant timestamps and values for all 'component-KPI' time series. This dataset is now ready for further anomaly detection within the specified time range.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp   cmdb_id                                           kpi_name        value                                      component_KPI
773778  1615359600      MG01           OSLinux-OSLinux_NETWORK_ens160_NETOutErr       0.0000      MG01-OSLinux-OSLinux_NETWORK_ens160_NETOutErr
773779  1615359600      MG01    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec  626479.0000  MG01-OSLinux-OSLinux_NETWORK_ens160_NETKBTotal...
773780  1615359600      MG01         OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc       0.0000    MG01-OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc
773781  1615359600  Tomcat03                         OSLinux-CPU_CPU_CPUSysTime       0.8860                Tomcat03-OSLinux-CPU_CPU_CPUSysTime
773782  1615359600  Tomcat03                             OSLinux-CPU_CPU_CPUWio       0.1913                    Tomcat03-OSLinux-CPU_CPU_CPUWio
...            ...       ...                                                ...          ...                                                ...
800366  1615361400  Tomcat02         OSLinux-OSLinux_FILESYSTEM_-tmp_FSCapacity       1.4017  Tomcat02-OSLinux-OSLinux_FILESYSTEM_-tmp_FSCap...
800367  1615361400  Tomcat02  OSLinux-OSLinux_FILESYSTEM_-tmp_FSInodeUsedPer...       0.0204  Tomcat02-OSLinux-OSLinux_FILESYSTEM_-tmp_FSIno...
800368  1615361400  Tomcat02      OSLinux-OSLinux_FILESYSTEM_-tomcat_FSCapacity      58.5351  Tomcat02-OSLinux-OSLinux_FILESYSTEM_-tomcat_FS...
800369  1615361400  Tomcat02  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSCapacity       0.4268  Tomcat02-OSLinux-OSLinux_FILESYSTEM_-cmbc_admi...
800370  1615361400  Tomcat02                    OSLinux-CPU_CPU-3_SingleCpuidle      96.7565           Tomcat02-OSLinux-CPU_CPU-3_SingleCpuidle

[26456 rows x 5 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Merge the filtered dataset with the P95 thresholds to enable anomaly detection
filtered_with_thresholds = filtered_time_range_df.merge(
    p95_thresholds, on='component_KPI', how='left'
)

# Identify anomalies where the value exceeds the P95 threshold
filtered_with_thresholds['is_anomaly'] = filtered_with_thresholds['value'] > filtered_with_thresholds['P95_Threshold']

# Filter only the anomalies for further analysis
anomalies = filtered_with_thresholds[filtered_with_thresholds['is_anomaly']]

anomalies

```
Out[4]:
```
Anomaly detection identified 1,011 data points in the filtered dataset (March 10, 2021, from 15:00 to 15:30) where the values exceeded the global P95 thresholds for their respective 'component-KPI' time series. These anomalies are flagged for further analysis to investigate potential issues in the system.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp   cmdb_id                                          kpi_name         value                                      component_KPI  P95_Threshold  is_anomaly
3      1615359600  Tomcat03                        OSLinux-CPU_CPU_CPUSysTime  8.860000e-01                Tomcat03-OSLinux-CPU_CPU_CPUSysTime   8.653000e-01        True
15     1615359600      MG02                       OSLinux-CPU_CPU_CPUUserTime  2.667670e+01                   MG02-OSLinux-CPU_CPU_CPUUserTime   2.655288e+01        True
26     1615359600      MG02         JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  1.214539e+09     MG02-JVM-Memory_7779_JVM_Memory_HeapMemoryUsed   1.192605e+09        True
146    1615359600   Redis01   OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum  1.104000e+03  Redis01-OSLinux-OSLinux_NETWORK_NETWORK_TotalT...   6.110000e+02        True
178    1615359600  apache01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite  1.520000e+01  apache01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-s...   1.306667e+01        True
...           ...       ...                                               ...           ...                                                ...            ...         ...
26409  1615361400  Tomcat02                        OSLinux-CPU_CPU_CPUCpuUtil  2.911360e+01                Tomcat02-OSLinux-CPU_CPU_CPUCpuUtil   2.888414e+01        True
26411  1615361400   Mysql02                       OSLinux-CPU_CPU_CPUUserTime  3.756337e+00                Mysql02-OSLinux-CPU_CPU_CPUUserTime   3.025633e+00        True
26412  1615361400   Mysql02                        OSLinux-CPU_CPU_CPUSysTime  2.227209e+00                 Mysql02-OSLinux-CPU_CPU_CPUSysTime   1.837115e+00        True
26416  1615361400   Mysql02                      Mysql-MySQL_3306_Select Scan  2.706670e+01               Mysql02-Mysql-MySQL_3306_Select Scan   2.491668e+01        True
26426  1615361400   Mysql02            Mysql-MySQL_3306_Table open cache hits  5.017667e+02     Mysql02-Mysql-MySQL_3306_Table open cache hits   3.381744e+02        True

[1011 rows x 7 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Group consecutive anomalies for each 'component-KPI' time series
anomalies['timestamp_diff'] = anomalies.groupby('component_KPI')['timestamp'].diff().fillna(0)
anomalies['is_consecutive'] = anomalies['timestamp_diff'] <= 60  # Assuming anomalies within 60 seconds are consecutive

# Assign fault group IDs for consecutive anomalies
anomalies['fault_group'] = (
    anomalies.groupby('component_KPI')['is_consecutive']
    .cumsum()
    .astype(int)
)

# Calculate extremal values for each fault group
fault_extremes = anomalies.groupby(['component_KPI', 'fault_group'])['value'].max().reset_index()
fault_extremes.columns = ['component_KPI', 'fault_group', 'extremal_value']

# Merge extremal values back into the anomalies DataFrame
anomalies_with_extremes = anomalies.merge(fault_extremes, on=['component_KPI', 'fault_group'], how='left')

# Filter out isolated noise spikes where the threshold breach is <= 50% of the extremal value
significant_faults = anomalies_with_extremes[
    anomalies_with_extremes['value'] > 0.5 * anomalies_with_extremes['extremal_value']
]

significant_faults

```
Out[5]:
```
After grouping consecutive anomalies for each 'component-KPI' time series and filtering out isolated noise spikes, 981 significant faults were identified in the filtered dataset. These faults represent meaningful threshold breaches and are ready for further investigation to determine their impact on the system.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp   cmdb_id                                          kpi_name         value                                      component_KPI  ...  is_anomaly  timestamp_diff  is_consecutive  fault_group  extremal_value
0     1615359600  Tomcat03                        OSLinux-CPU_CPU_CPUSysTime  8.860000e-01                Tomcat03-OSLinux-CPU_CPU_CPUSysTime  ...        True             0.0            True            1    9.653000e-01
1     1615359600      MG02                       OSLinux-CPU_CPU_CPUUserTime  2.667670e+01                   MG02-OSLinux-CPU_CPU_CPUUserTime  ...        True             0.0            True            1    2.667670e+01
2     1615359600      MG02         JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  1.214539e+09     MG02-JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  ...        True             0.0            True            1    1.214539e+09
3     1615359600   Redis01   OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum  1.104000e+03  Redis01-OSLinux-OSLinux_NETWORK_NETWORK_TotalT...  ...        True             0.0            True            1    1.104000e+03
4     1615359600  apache01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite  1.520000e+01  apache01-OSLinux-OSLinux_LOCALDISK_LOCALDISK-s...  ...        True             0.0            True            1    1.986670e+01
...          ...       ...                                               ...           ...                                                ...  ...         ...             ...             ...          ...             ...
1006  1615361400  Tomcat02                        OSLinux-CPU_CPU_CPUCpuUtil  2.911360e+01                Tomcat02-OSLinux-CPU_CPU_CPUCpuUtil  ...        True           660.0           False            1    2.911360e+01
1007  1615361400   Mysql02                       OSLinux-CPU_CPU_CPUUserTime  3.756337e+00                Mysql02-OSLinux-CPU_CPU_CPUUserTime  ...        True            60.0            True            5    3.756337e+00
1008  1615361400   Mysql02                        OSLinux-CPU_CPU_CPUSysTime  2.227209e+00                 Mysql02-OSLinux-CPU_CPU_CPUSysTime  ...        True            60.0            True            4    2.227209e+00
1009  1615361400   Mysql02                      Mysql-MySQL_3306_Select Scan  2.706670e+01               Mysql02-Mysql-MySQL_3306_Select Scan  ...        True           360.0           False            2    2.965570e+01
1010  1615361400   Mysql02            Mysql-MySQL_3306_Table open cache hits  5.017667e+02     Mysql02-Mysql-MySQL_3306_Table open cache hits  ...        True            60.0            True            3    5.017667e+02

[981 rows x 11 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Define fault levels based on component naming conventions
def determine_fault_level(component):
    if component.startswith("docker") or component.startswith("Tomcat"):
        return "container"
    elif component.startswith("apache") or component.startswith("MG") or component.startswith("IG"):
        return "service"
    elif component.startswith("Mysql") or component.startswith("Redis"):
        return "node"
    else:
        return "unknown"

# Add a column for fault level
significant_faults['fault_level'] = significant_faults['cmdb_id'].apply(determine_fault_level)

# Calculate the maximum deviation from the threshold for each fault level
significant_faults['deviation'] = significant_faults['value'] - significant_faults['P95_Threshold']
max_deviation_by_level = significant_faults.groupby('fault_level')['deviation'].max().reset_index()
max_deviation_by_level.columns = ['fault_level', 'max_deviation']

max_deviation_by_level

```
Out[6]:
```
The analysis of identified faults shows that they correspond to three levels: container, node, and service. The maximum deviation from the threshold for each level is as follows:

- **Container level**: 313,554.0
- **Node level**: 7,570,467.2
- **Service level**: 41,168,687.6

The predominant fault level is the **service level**, as it exhibits the highest maximum deviation from the threshold.

The original code execution output of IPython Kernel is also provided below for reference:

fault_level  max_deviation
0   container       313554.0
1        node      7570467.2
2     service     41168687.6```
```

In [ ]:
# In[7]:

# Load the trace_span.csv file for March 10, 2021
trace_file_path = 'dataset/Bank/telemetry/2021_03_10/trace/trace_span.csv'
trace_span_df = pd.read_csv(trace_file_path)

# Convert the timestamp in trace_span_df to seconds for consistency with the time range
trace_span_df['timestamp_seconds'] = (trace_span_df['timestamp'] / 1000).astype(int)

# Filter the trace data within the specified time range (15:00 to 15:30)
trace_filtered_df = trace_span_df[
    (trace_span_df['timestamp_seconds'] >= start_time) & 
    (trace_span_df['timestamp_seconds'] <= end_time)
]

# Focus on traces that include the identified service-level faulty components
service_faulty_components = significant_faults[significant_faults['fault_level'] == 'service']['cmdb_id'].unique()
trace_service_faults = trace_filtered_df[trace_filtered_df['cmdb_id'].isin(service_faulty_components)]

# Identify the most downstream faulty service for each trace
# A downstream service is identified as the one with no child spans (i.e., its span_id is not a parent_id)
downstream_faults = trace_service_faults[
    ~trace_service_faults['span_id'].isin(trace_filtered_df['parent_id'])
]

downstream_faults

```
Out[7]:
```
The analysis of the `trace_span.csv` file for March 10, 2021, within the time range of 15:00 to 15:30, identified 37,955 downstream spans associated with the service-level faulty components. These spans represent the most downstream faulty services in the traces and are critical for understanding the propagation of failures in the system.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp cmdb_id             parent_id                                            span_id                  trace_id  duration  timestamp_seconds
14350275  1615359600087    MG01  21031000016623499327  369-bcou-dle-way1-c514cf30-43410@0824-2f0e4898...  gw0120210310145957021064         7         1615359600
14350282  1615359600137    MG01  21031000016623499408  369-bcou-dle-way1-c514cf30-43410@0824-2f0e4898...  gw0120210310150000021106         5         1615359600
14350315  1615359600035    MG01  21031000016623499266  369-bcou-dle-way1-c514cf30-43410@0824-2f0e4898...  gw0120210310145958914039         6         1615359600
14350317  1615359600121    MG01  21031000016623499390  369-bcou-dle-way1-c514cf30-43410@0824-2f0e4898...  gw0120210310145958914030         8         1615359600
14350339  1615359600238    MG02  21031000016623652802  369-bcou-dle-way1-c514cf31-96790@1019-2f0e4898...  gw0120210310150000021108         5         1615359600
...                 ...     ...                   ...                                                ...                       ...       ...                ...
15463487  1615361400958    MG01  21031000016626528983  369-bcou-dle-way1-c514cf30-43410@0824-2f0e4898...  gw0120210310152958050224         5         1615361400
15463493  1615361400978    MG02  21031000016626462569  369-bcou-dle-way1-c514cf31-96790@1019-2f0e4898...  gw0120210310153000050270         5         1615361400
15463617  1615361400198    MG01  21031000016626529100  369-bcou-dle-way1-c514cf30-43410@0824-2f0e4898...  gw0120210310152958942950      1011         1615361400
15463683  1615361400556    MG02  21031000016626462964  369-bcou-dle-way1-c514cf31-96790@1019-2f0e4898...  gw0120210310152959942977      1020         1615361400
15463777  1615361400572    MG02  21031000016626463031  369-bcou-dle-way1-c514cf31-96790@1019-2f0e4898...  gw0120210310153000050261      1026         1615361400

[37955 rows x 7 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

# Identify the first occurrence of a faulty service in the downstream spans for each failure
first_occurrences = downstream_faults.groupby('trace_id').first().reset_index()

# Correlate these occurrences with the failure timestamps
root_cause_analysis = first_occurrences[['timestamp_seconds', 'cmdb_id', 'trace_id']]
root_cause_analysis.columns = ['occurrence_time', 'root_cause_component', 'trace_id']

root_cause_analysis

```
Out[8]:
```
The analysis identified 25,730 occurrences of faulty services in the downstream spans within the specified time range of March 10, 2021, from 15:00 to 15:30. Each occurrence is correlated with its failure timestamp and the root cause component, providing a detailed mapping of when and where each failure originated. This information is crucial for pinpointing the root cause components and their impact on the system.

The original code execution output of IPython Kernel is also provided below for reference:

occurrence_time root_cause_component                  trace_id
0           1615359600                 MG02  gw0120210310145956913995
1           1615359600                 MG01  gw0120210310145957021064
2           1615359600                 MG01  gw0120210310145957021066
3           1615359602                 MG01  gw0120210310145958021083
4           1615359600                 MG01  gw0120210310145958914030
...                ...                  ...                       ...
25725       1615361400                 MG02  gw0120210310153000050270
25726       1615361400                 MG01  gw0120210310153000942986
25727       1615361400                 MG02  gw0120210310153000942989
25728       1615361400                 MG01  gw0120210310153000942990
25729       1615361400                 MG02  gw0120210310153000942991

[25730 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

# Group the identified occurrences of faulty services by failure timestamps
grouped_occurrences = root_cause_analysis.groupby('occurrence_time')

# Select the most downstream faulty service within each group as the root cause component
most_downstream_faults = grouped_occurrences.first().reset_index()

most_downstream_faults

```
Out[9]:
```
The analysis grouped 1,801 occurrences of faulty services by failure timestamps and identified the most downstream faulty service within each group as the root cause component. Each failure is now associated with a specific root cause component and its occurrence time, providing a clear understanding of the origin of each failure for further investigation and resolution.

The original code execution output of IPython Kernel is also provided below for reference:

occurrence_time root_cause_component                  trace_id
0          1615359600                 MG02  gw0120210310145956913995
1          1615359601                 MG01  gw0120210310150000021105
2          1615359602                 MG01  gw0120210310145958021083
3          1615359603                 MG01  gw0120210310150003021151
4          1615359604                 MG02  gw0120210310150002021145
...               ...                  ...                       ...
1796       1615361396                 MG02  gw0120210310152954050121
1797       1615361397                 MG02  gw0120210310152955050155
1798       1615361398                 MG01  gw0120210310152956050170
1799       1615361399                 MG01  gw0120210310152957942927
1800       1615361400                 MG01  gw0120210310152958050215

[1801 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```